# Assamese OCR - Sentence Model Training

Train a CRNN model for Assamese sentence-level OCR using CTC loss.

## Setup Requirements
1. **GPU**: Settings → Accelerator → GPU T4 x2
2. **Dataset**: Add your `as-wiki-2021.txt` via "+ Add Data"
3. **Checkpoints** (optional): Add existing `.pth` files via "+ Add Data"

---

## 1. Clone Repository

In [ ]:
import os
import shutil

# Ensure working directory exists
os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

# Clone fresh
REPO = 'Assamese_OCR'
if os.path.exists(REPO):
    shutil.rmtree(REPO)

!git clone https://github.com/rishav660/Assamese_OCR.git

# Navigate to backend
os.chdir(f'/kaggle/working/{REPO}/django_backend')
print(f'✅ Working directory: {os.getcwd()}')

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Setup Data

Update `WIKI_PATH` to match your dataset location.

In [ ]:
# Update this path
WIKI_PATH = '/kaggle/input/assamese-wiki/as-wiki-2021.txt'

if not os.path.exists(WIKI_PATH):
    print(f'❌ Wiki file not found: {WIKI_PATH}')
    print('Add dataset via "+ Add Data" button')
else:
    print(f'✅ Wiki file found ({os.path.getsize(WIKI_PATH) / 1e6:.1f} MB)')

# Create directories
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# Link wiki file
if os.path.exists(WIKI_PATH) and not os.path.exists('data/as-wiki-2021.txt'):
    os.symlink(WIKI_PATH, 'data/as-wiki-2021.txt')
    print('✅ Wiki file linked')

In [ ]:
# Optional: Copy existing checkpoints
CHECKPOINT_INPUT = '/kaggle/input/assamese-ocr-checkpoints'

if os.path.exists(CHECKPOINT_INPUT):
    for f in os.listdir(CHECKPOINT_INPUT):
        if f.endswith('.pth'):
            shutil.copy2(os.path.join(CHECKPOINT_INPUT, f), f'checkpoints/{f}')
            print(f'Copied: {f}')
else:
    print('No checkpoint input found - training from scratch')

## 4. Generate Training Splits

In [ ]:
# Create split directories
for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    path = f'data/{split}'
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(f'{path}/images', exist_ok=True)
    os.makedirs(f'{path}/labels', exist_ok=True)

print('✅ Directories ready')

In [ ]:
!python build_real_sentence_splits.py \
    --input data/as-wiki-2021.txt \
    --train-output data/train_real_sentences \
    --val-output data/val_real_sentences \
    --test-output data/test_real_sentences \
    --train-count 50000 \
    --val-count 5000 \
    --test-count 2000 \
    --seed 42

In [ ]:
# Verify splits
for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    img_dir = f'data/{split}/images'
    n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    print(f'{split}: {n} images')

## 5. Train Model

**Auto-resume enabled**: The script saves a checkpoint after every epoch to `latest_model_sentences.pth`.  
If Kaggle disconnects, just re-run this cell — it will automatically resume from the last completed epoch.

In [ ]:
# Check for existing checkpoints
latest_ckpt = 'checkpoints/latest_model_sentences.pth'
if os.path.exists(latest_ckpt):
    import torch
    ckpt = torch.load(latest_ckpt, map_location='cpu')
    if isinstance(ckpt, dict) and 'epoch' in ckpt:
        print(f'✅ Found checkpoint from epoch {ckpt["epoch"] + 1}')
        print(f'   Best val loss: {ckpt.get("best_val_loss", "N/A"):.4f}')
        print(f'   Training will resume automatically')
    else:
        print('✅ Found checkpoint (legacy format)')
else:
    print('ℹ️  No checkpoint found - training from scratch')

In [ ]:
!python train_sentence_model.py \
    --train-img-dir data/train_real_sentences/images \
    --train-label-file data/train_real_sentences/labels/labels.txt \
    --val-img-dir data/val_real_sentences/images \
    --val-label-file data/val_real_sentences/labels/labels.txt \
    --epochs 25 \
    --batch-size 32 \
    --learning-rate 0.0001 \
    --best-checkpoint checkpoints/best_model_sentences.pth \
    --final-checkpoint checkpoints/final_model_sentences.pth \
    --latest-checkpoint checkpoints/latest_model_sentences.pth \
    --resume \
    --plot-out training_curve.png

In [ ]:
# View training curve
from IPython.display import Image
if os.path.exists('training_curve.png'):
    display(Image('training_curve.png'))

## 6. Evaluate Model

In [ ]:
# Validation set - greedy decode
!python evaluate.py \
    --img-dir data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --num-samples 10

In [ ]:
# Validation set - beam search
!python evaluate.py \
    --img-dir data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --beam-width 10 \
    --num-samples 10

In [ ]:
# Test set evaluation
!python evaluate.py \
    --img-dir data/test_real_sentences/images \
    --label-file data/test_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --num-samples 10 \
    --output test_results.tsv

## 7. Download Results

All files in `/kaggle/working/` are saved to the Output tab.

In [ ]:
!ls -lh checkpoints/*.pth 2>/dev/null || echo 'No checkpoints'
!ls -lh *.png *.tsv 2>/dev/null || echo 'No results'